# LeetCode #134 — Gas Station: tham lam một lượt với hai nhận xét then chốt

**Mức độ:** Medium · **Chủ đề:** Array, Greedy

Skeleton trên LeetCode:

```python
class Solution:
    def canCompleteCircuit(self, gas: List[int], cost: List[int]) -> int:
```

Mục tiêu của notebook:

1. Đổi bài toán thành mảng chênh lệch `diff[i] = gas[i] − cost[i]`.
2. Nhận xét 1: nếu `sum(diff) < 0` thì **không** có lời giải; ngược lại **luôn** có.
3. Nhận xét 2: nếu xuất phát từ `s` mà hết xăng trước khi tới `j + 1`, thì **mọi** điểm giữa `s..j` cũng thất bại → nhảy thẳng tới `j + 1`.
4. Liên hệ với **tổng tiền tố nhỏ nhất** (cách nhìn khác cho cùng thuật toán).

# 1. Bài toán là gì?

Có `n` trạm xăng trên một **đường vòng**. Trạm `i` có `gas[i]` lít; đi từ trạm `i` tới trạm `i + 1` tốn `cost[i]` lít. Xe có bình **vô hạn**, bắt đầu với bình **rỗng** tại một trạm nào đó.

Trả về chỉ số trạm xuất phát để đi **hết một vòng** theo chiều kim đồng hồ, hoặc `-1` nếu không thể. Đề đảm bảo nếu có lời giải thì nó **duy nhất**.

**Constraints (theo đề):**

- `n == gas.length == cost.length`
- `1 <= n <= 10^5`
- `0 <= gas[i], cost[i] <= 10^4`

---

## Ví dụ

```text
gas  = [1, 2, 3, 4, 5]
cost = [3, 4, 5, 1, 2]   ->  3

gas  = [2, 3, 4]
cost = [3, 4, 3]         ->  -1
```

Ví dụ 1, xuất phát ở trạm 3: bình `0 + 4 = 4`, đi tốn 1 → `3`; trạm 4: `3 + 5 = 8`, tốn 2 → `6`; trạm 0: `7`, tốn 3 → `4`; trạm 1: `6`, tốn 4 → `2`; trạm 2: `5`, tốn 5 → `0`. Về trạm 3 — thành công.

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Thuật toán tham lam (greedy)
2. Tổng tiền tố (prefix sum) và tổng chạy

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Thuật toán tham lam (greedy)

### Tham lam là gì?

Thuật toán tham lam xây lời giải bằng cách ở **mỗi bước chọn phương án tốt nhất trước mắt**, không bao giờ quay lại sửa. Nhanh và đơn giản — nhưng **không phải lúc nào cũng đúng**.

### Ví dụ đúng: đổi tiền với mệnh giá 1, 2, 5, 10

Luôn lấy tờ **lớn nhất** không vượt số tiền còn lại.

### Ví dụ sai: mệnh giá 1, 3, 4, cần 6

Tham lam: `4 + 1 + 1` (3 tờ). Tối ưu: `3 + 3` (2 tờ). Chọn tốt nhất trước mắt dẫn tới kết quả tệ hơn.

In [1]:
def doi_tien_tham_lam(menh_gia, so_tien):
    chon = []
    for m in sorted(menh_gia, reverse=True):
        while so_tien >= m:
            so_tien -= m
            chon.append(m)
    return chon

def doi_tien_toi_uu(menh_gia, so_tien):           # DP để đối chiếu
    INF = float("inf")
    dp = [0] + [INF] * so_tien
    for x in range(1, so_tien + 1):
        for m in menh_gia:
            if m <= x:
                dp[x] = min(dp[x], dp[x - m] + 1)
    return dp[so_tien]

print("mệnh giá [1,2,5,10], 18:", doi_tien_tham_lam([1, 2, 5, 10], 18), "| tối ưu:", doi_tien_toi_uu([1, 2, 5, 10], 18), "tờ")
print("mệnh giá [1,3,4], 6:", doi_tien_tham_lam([1, 3, 4], 6), "| tối ưu:", doi_tien_toi_uu([1, 3, 4], 6), "tờ")

mệnh giá [1,2,5,10], 18: [10, 5, 2, 1] | tối ưu: 4 tờ
mệnh giá [1,3,4], 6: [4, 1, 1] | tối ưu: 2 tờ


### Làm sao biết tham lam đúng?

Phải **chứng minh**. Hai kỹ thuật hay dùng:

1. **Lập luận trao đổi (exchange argument)**: giả sử có lời giải tối ưu khác lựa chọn tham lam ở bước đầu; chỉ ra có thể **đổi** nó sang lựa chọn tham lam mà không tệ đi.
2. **Bất biến "luôn dẫn đầu"**: sau mỗi bước, lời giải tham lam không kém bất kỳ lời giải nào khác.

Và luôn **thử phản ví dụ** bằng brute force trên dữ liệu nhỏ.

### Ví dụ kinh điển: chọn nhiều cuộc họp nhất không trùng giờ

Chọn cuộc họp **kết thúc sớm nhất** trước — để lại nhiều thời gian nhất cho phần còn lại (lập luận trao đổi: thay cuộc họp đầu của lời giải tối ưu bằng cuộc kết thúc sớm nhất vẫn không chồng lấn).

In [2]:
def nhieu_cuoc_hop(hop):
    chon, ket_thuc = [], float("-inf")
    for s, e in sorted(hop, key=lambda x: x[1]):     # theo giờ kết thúc
        if s >= ket_thuc:
            chon.append((s, e))
            ket_thuc = e
    return chon

hop_demo = [(1, 4), (3, 5), (0, 6), (5, 7), (3, 9), (5, 9), (6, 10), (8, 11), (8, 12), (2, 14), (12, 16)]
def theo_bat_dau(hop):                               # tiêu chí SAI để so sánh
    chon, ket_thuc = [], float("-inf")
    for s, e in sorted(hop):
        if s >= ket_thuc:
            chon.append((s, e)); ket_thuc = e
    return chon

print("kết thúc sớm nhất trước:", nhieu_cuoc_hop(hop_demo), "→", len(nhieu_cuoc_hop(hop_demo)), "cuộc")
print("bắt đầu sớm nhất trước (sai):", theo_bat_dau(hop_demo), "→", len(theo_bat_dau(hop_demo)), "cuộc")

kết thúc sớm nhất trước: [(1, 4), (5, 7), (8, 11), (12, 16)] → 4 cuộc
bắt đầu sớm nhất trước (sai): [(0, 6), (6, 10), (12, 16)] → 3 cuộc


### Lỗi hay gặp

- Tin tham lam đúng mà không chứng minh / không thử phản ví dụ.
- Chọn sai tiêu chí tham lam (bắt đầu sớm nhất, ngắn nhất… thay vì kết thúc sớm nhất).
- Quên sắp xếp theo tiêu chí trước khi duyệt.

## 2.2 Tổng tiền tố (prefix sum) và tổng chạy

### Tổng tiền tố là gì?

Với mảng `a`, mảng tổng tiền tố `P` có `P[0] = 0` và `P[i] = a[0] + … + a[i−1]`. Khi đó tổng một đoạn bất kỳ:

```text
a[l] + … + a[r] = P[r + 1] − P[l]          (O(1) sau khi dựng P trong O(n))
```

In [3]:
from itertools import accumulate

a_ps = [3, -1, 4, 1, -5, 9]
P_ps = [0] + list(accumulate(a_ps))
print("a =", a_ps)
print("P =", P_ps)
l_ps, r_ps = 1, 3
print(f"tổng a[{l_ps}..{r_ps}] = P[{r_ps + 1}] - P[{l_ps}] = {P_ps[r_ps + 1] - P_ps[l_ps]} =", sum(a_ps[l_ps:r_ps + 1]))

a = [3, -1, 4, 1, -5, 9]
P = [0, 3, 2, 6, 7, 2, 11]
tổng a[1..3] = P[4] - P[1] = 4 = 4


### Tổng chạy và điểm "thấp nhất"

Trong nhiều bài, ta chỉ cần **tổng chạy** (running sum) vừa duyệt vừa cộng. Một quan sát hay dùng cho mảng **vòng tròn** có tổng toàn bộ `≥ 0`: nếu tổng chạy **xuống thấp nhất** tại vị trí `k`, thì xuất phát ngay sau `k` (đi vòng) mọi tổng chạy đều `≥ 0` — không có đoạn nào làm tổng bị âm.

In [4]:
b_ps = [1, -3, 2, -1, 3, -1]                 # tổng = 1 >= 0
tong_chay, thap_nhat, vi_tri = 0, 0, -1
for i, v in enumerate(b_ps):
    tong_chay += v
    if tong_chay < thap_nhat:
        thap_nhat, vi_tri = tong_chay, i
    print(f"   i={i} a[i]={v:>2} tổng chạy={tong_chay:>2}")
print("tổng chạy thấp nhất", thap_nhat, "ngay sau vị trí", vi_tri, "→ xuất phát từ", vi_tri + 1)
start_ps = vi_tri + 1
quay_vong = b_ps[start_ps:] + b_ps[:start_ps]
print("xuất phát từ 0 thì các tổng chạy:", list(accumulate(b_ps)), "(có số âm)")
print("xuất phát từ đó, các tổng chạy:", list(accumulate(quay_vong)))

   i=0 a[i]= 1 tổng chạy= 1
   i=1 a[i]=-3 tổng chạy=-2
   i=2 a[i]= 2 tổng chạy= 0
   i=3 a[i]=-1 tổng chạy=-1
   i=4 a[i]= 3 tổng chạy= 2
   i=5 a[i]=-1 tổng chạy= 1
tổng chạy thấp nhất -2 ngay sau vị trí 1 → xuất phát từ 2
xuất phát từ 0 thì các tổng chạy: [1, -2, 0, -1, 2, 1] (có số âm)
xuất phát từ đó, các tổng chạy: [2, 1, 4, 3, 4, 1]


Lỗi hay gặp: lệch chỉ số giữa `P[i]` và `a[i]`; quên `P[0] = 0`; dùng `sum(a[l:r+1])` trong vòng lặp (thành `O(n²)`).

## 2.3 Áp dụng vào bài này

Đặt `diff[i] = gas[i] − cost[i]`. Nếu tổng `diff < 0` thì không có đáp án; ngược lại điểm xuất phát nằm ngay sau chỗ **tổng chạy thấp nhất** — đúng quan sát "tổng chạy" ở trên. Lời giải **tham lam** một lượt: hễ bình xăng âm thì bỏ hết đoạn đã đi, xuất phát lại từ trạm kế tiếp.

# 3. Baseline: thử mọi điểm xuất phát

`O(n²)` — quá chậm với `n = 10^5`, nhưng dùng để kiểm tra.

In [5]:
from typing import List
import random, time

def can_complete_brute(gas: List[int], cost: List[int]) -> int:
    n = len(gas)
    for s in range(n):
        tank = 0
        for k in range(n):
            i = (s + k) % n
            tank += gas[i] - cost[i]
            if tank < 0:
                break
        else:
            return s
    return -1

print(can_complete_brute([1, 2, 3, 4, 5], [3, 4, 5, 1, 2]), can_complete_brute([2, 3, 4], [3, 4, 3]))

3 -1


# 4. Hai nhận xét then chốt

Đặt `diff[i] = gas[i] − cost[i]` (lượng xăng "lời" khi đi qua đoạn `i → i+1`).

**Nhận xét 1 — điều kiện tồn tại:** đi hết vòng thì tổng xăng nhận được trừ tổng tiêu thụ là `sum(diff)`. Nếu `sum(diff) < 0` thì chắc chắn thất bại. Ngược lại (`sum(diff) ≥ 0`) **luôn** tồn tại điểm xuất phát (chứng minh ở mục 8).

**Nhận xét 2 — bỏ qua cả đoạn:** xuất phát tại `s`, giả sử bình **lần đầu** âm khi tới sau trạm `j` (tức `diff[s] + … + diff[j] < 0`). Với mọi `k` trong `s+1..j`: khi xuất phát từ `s` và tới `k`, bình đang có `diff[s] + … + diff[k−1] ≥ 0` (vì chưa âm). Xuất phát từ `k` thì bình bắt đầu bằng 0 — **không hơn** — nên cũng sẽ âm trước khi qua `j`. Vậy bỏ qua mọi `k ≤ j`, thử tiếp từ `j + 1`.

In [6]:
def can_complete(gas: List[int], cost: List[int]) -> int:
    total = tank = start = 0
    for i in range(len(gas)):
        d = gas[i] - cost[i]
        total += d
        tank += d
        if tank < 0:
            start = i + 1
            tank = 0
    return start if total >= 0 else -1

print(can_complete([1, 2, 3, 4, 5], [3, 4, 5, 1, 2]), can_complete([2, 3, 4], [3, 4, 3]))

3 -1


Chỉ **một lượt** qua mảng, không cần đi vòng lại: nếu `total ≥ 0` thì `start` cuối cùng chắc chắn đi hết vòng.

# 5. Mô phỏng

In [7]:
def trace(gas, cost):
    total = tank = start = 0
    print(f"{'i':>2} {'gas':>4} {'cost':>4} {'diff':>5} {'tank':>5} {'start':>5}")
    for i in range(len(gas)):
        d = gas[i] - cost[i]
        total += d
        tank += d
        note = ""
        if tank < 0:
            note = f"  tank âm -> bỏ {start}..{i}, start = {i + 1}"
            start = i + 1
            tank = 0
        print(f"{i:>2} {gas[i]:>4} {cost[i]:>4} {d:>+5} {tank:>5} {start:>5}{note}")
    print(f"total = {total} -> {'kết quả ' + str(start) if total >= 0 else '-1'}")

trace([1, 2, 3, 4, 5], [3, 4, 5, 1, 2])

 i  gas cost  diff  tank start
 0    1    3    -2     0     1  tank âm -> bỏ 0..0, start = 1
 1    2    4    -2     0     2  tank âm -> bỏ 1..1, start = 2
 2    3    5    -2     0     3  tank âm -> bỏ 2..2, start = 3
 3    4    1    +3     3     3
 4    5    2    +3     6     3
total = 0 -> kết quả 3


# 6. Cách nhìn khác: tổng tiền tố nhỏ nhất

Gọi `P[i] = diff[0] + … + diff[i]`. Xuất phát tại `s` khả thi ⇔ mọi tổng "từ `s` đi một vòng" đều ≥ 0. Nếu chọn `s` ngay **sau** vị trí có `P` **nhỏ nhất**, mọi tổng từ `s` trở đi là `P[i] − P[min] ≥ 0` (phần trước vòng), và phần quay vòng được bù bởi `total ≥ 0`.

In [8]:
def can_complete_min_prefix(gas, cost):
    total, min_p, min_i = 0, float("inf"), -1
    for i in range(len(gas)):
        total += gas[i] - cost[i]
        if total < min_p:
            min_p, min_i = total, i
    return (min_i + 1) % len(gas) if total >= 0 else -1

print(can_complete_min_prefix([1, 2, 3, 4, 5], [3, 4, 5, 1, 2]), can_complete_min_prefix([2, 3, 4], [3, 4, 3]))

3 -1


Khi đề có lời giải **duy nhất**, hai cách cho cùng kết quả. Với input có nhiều điểm khả thi (đề chính thức không có), hai cách có thể chọn điểm khác nhau — nhưng điểm nào cũng khả thi (kiểm tra ở mục 7).

# 7. Phiên bản đúng format LeetCode

In [9]:
class Solution:
    def canCompleteCircuit(self, gas: List[int], cost: List[int]) -> int:
        total = tank = start = 0
        for i in range(len(gas)):
            d = gas[i] - cost[i]
            total += d
            tank += d
            if tank < 0:
                start = i + 1
                tank = 0
        return start if total >= 0 else -1

In [10]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<18} PASS  {actual}")


def feasible(gas, cost, s):
    tank = 0
    for k in range(len(gas)):
        i = (s + k) % len(gas)
        tank += gas[i] - cost[i]
        if tank < 0:
            return False
    return True


sol = Solution()
check("Ví dụ 1", sol.canCompleteCircuit([1, 2, 3, 4, 5], [3, 4, 5, 1, 2]), 3)
check("Ví dụ 2", sol.canCompleteCircuit([2, 3, 4], [3, 4, 3]), -1)
check("Một trạm đủ", sol.canCompleteCircuit([5], [4]), 0)
check("Một trạm thiếu", sol.canCompleteCircuit([4], [5]), -1)
check("Vừa đủ", sol.canCompleteCircuit([3, 1, 1], [1, 2, 2]), 0)

random.seed(134)
for _ in range(5000):
    n = random.randint(1, 8)
    gas = [random.randint(0, 6) for _ in range(n)]
    cost = [random.randint(0, 6) for _ in range(n)]
    r = sol.canCompleteCircuit(gas, cost)
    exists = any(feasible(gas, cost, s) for s in range(n))
    assert (r == -1) == (not exists)
    if r != -1:
        assert feasible(gas, cost, r)
    assert (can_complete_min_prefix(gas, cost) == -1) == (not exists)
    if exists:
        assert feasible(gas, cost, can_complete_min_prefix(gas, cost))
print("5000 test (kể cả có nhiều lời giải): điểm trả về luôn khả thi; -1 đúng khi vô nghiệm")

n = 10**5
gas = [random.randint(0, 10**4) for _ in range(n)]
cost = gas[:]; random.shuffle(cost)
t0 = time.time(); r = sol.canCompleteCircuit(gas, cost); t1 = time.time()
print(f"n = 10^5: kết quả {r}, khả thi? {feasible(gas, cost, r)}, {t1 - t0:.3f}s")

Ví dụ 1            PASS  3
Ví dụ 2            PASS  -1
Một trạm đủ        PASS  0
Một trạm thiếu     PASS  -1
Vừa đủ             PASS  0


5000 test (kể cả có nhiều lời giải): điểm trả về luôn khả thi; -1 đúng khi vô nghiệm

n = 10^5: kết quả 68730, khả thi? True, 0.042s


Test ngẫu nhiên có thể có **nhiều** lời giải (đề đảm bảo duy nhất chỉ trong test chính thức), nên ta kiểm tra "điểm trả về khả thi" thay vì so với một đáp án cụ thể.

# 8. Vì sao đúng? Độ phức tạp?

Gọi `start` cuối cùng là `s`. Giả sử `total ≥ 0`; ta chứng minh xuất phát tại `s` thì bình không bao giờ âm.

- **Từ `s` tới `n − 1`:** sau lần đặt lại cuối, `tank` không âm lần nào nữa ⇒ mọi tổng `diff[s] + … + diff[i]` (với `i ≤ n−1`) đều ≥ 0. Gọi `B = diff[s] + … + diff[n−1] ≥ 0`.
- **Quay vòng qua `0 .. s−1`:** đoạn này bị chia thành các **khối**, mỗi khối kết thúc tại một lần đặt lại, nên mỗi khối có tổng **âm**, còn mọi **phần đầu** (không trọn) của một khối có tổng ≥ 0. Gọi `A = diff[0] + … + diff[s−1]` (tổng các khối, `A ≤ 0`). Vì `A + B = total ≥ 0` nên `B ≥ −A`.
- Một tiền tố bất kỳ `diff[0] + … + diff[k]` (với `k < s`) = (tổng các khối trọn vẹn trước nó) + (phần đầu của khối đang dở) ≥ (tổng các khối trọn vẹn) ≥ `A` (vì các khối còn lại đều âm). Vậy bình tại đó là `B + tiền tố ≥ B + A ≥ 0`.

```text
Time:  O(n), một lượt
Space: O(1)
```

# 9. Những lỗi dễ mắc

## Lỗi 1: Đặt `start = i` thay vì `i + 1`

In [11]:
def wrong_start_i(gas, cost):
    total = tank = start = 0
    for i in range(len(gas)):
        d = gas[i] - cost[i]
        total += d; tank += d
        if tank < 0:
            start = i                              # SAI: trạm i chính là chỗ thất bại
            tank = 0
    return start if total >= 0 else -1

r = wrong_start_i([1, 2, 3, 4, 5], [3, 4, 5, 1, 2])
print("Sai :", r, "| khả thi?", feasible([1, 2, 3, 4, 5], [3, 4, 5, 1, 2], r))

Sai : 2 | khả thi? False


## Lỗi 2: Không kiểm tra `total`

In [12]:
def wrong_no_total(gas, cost):
    tank = start = 0
    for i in range(len(gas)):
        tank += gas[i] - cost[i]
        if tank < 0:
            start = i + 1
            tank = 0
    return start % len(gas)                        # SAI: có thể vô nghiệm

r = wrong_no_total([2, 3, 4], [3, 4, 3])
print("Sai :", r, "| khả thi?", feasible([2, 3, 4], [3, 4, 3], r), "| đúng phải là -1")

Sai : 2 | khả thi? False | đúng phải là -1


## Lỗi 3: Chọn trạm có `gas[i] − cost[i]` lớn nhất

Trạm "lời" nhất chưa chắc là điểm xuất phát đúng — có thể phía sau nó là một chuỗi đoạn lỗ dài.

In [13]:
def wrong_max_diff(gas, cost):
    if sum(gas) < sum(cost):
        return -1
    return max(range(len(gas)), key=lambda i: gas[i] - cost[i])     # SAI

g, c = [5, 1, 1, 4], [1, 5, 2, 3]
r = wrong_max_diff(g, c)
print("Sai :", r, "| khả thi?", feasible(g, c, r), "| đúng:", sol.canCompleteCircuit(g, c))

Sai : 0 | khả thi? False | đúng: 3


## Lỗi 4: Mô phỏng vòng với `% n` cho mọi điểm (`O(n²)`)

Đúng nhưng quá chậm với `n = 10^5` (10^10 bước). Nhận xét 2 là thứ biến nó thành `O(n)`.

# 10. Mẹo nhận diện pattern

- **Bài toán vòng / "có điểm xuất phát nào"** → nghĩ tới tổng toàn bộ (điều kiện tồn tại) và **tổng tiền tố nhỏ nhất**.
- **"Nếu thất bại từ `s` tới `j` thì mọi điểm giữa cũng thất bại"** → bỏ qua cả đoạn, tham lam một lượt (giống Kadane #53 đặt lại tổng khi âm).

Bài liên quan: **#53 Maximum Subarray**, **#918 Maximum Sum Circular Subarray**, **#1567**, **#2202**.

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Chạy thuật toán cho `gas = [3, 1, 1]`, `cost = [1, 2, 2]` và `gas = [5, 8, 2, 8]`, `cost = [6, 5, 6, 6]`.

## Bài 2 — Điền chỗ trống

In [14]:
def exercise_2_can_complete(gas: List[int], cost: List[int]) -> int:
    total = tank = start = 0
    for i in range(len(gas)):
        # TODO
        pass
    return start if total >= 0 else -1


# assert exercise_2_can_complete([1, 2, 3, 4, 5], [3, 4, 5, 1, 2]) == 3
# assert exercise_2_can_complete([2, 3, 4], [3, 4, 3]) == -1
# assert exercise_2_can_complete([5], [4]) == 0

## Bài 3 — Biến thể: liệt kê **mọi** điểm xuất phát khả thi trong `O(n)`

Gợi ý nhanh: đặt `Q[t] = diff[0] + … + diff[t−1]` (`Q[0] = 0`, `Q[n] = total`). Xuất phát tại `s`, bình sau trạm `i ≥ s` là `Q[i+1] − Q[s]`, và sau trạm `j < s` (đã quay vòng) là `total − Q[s] + Q[j+1]`. Cả hai phải ≥ 0.

In [15]:
def exercise_3_all_starts(gas: List[int], cost: List[int]) -> List[int]:
    # TODO
    pass


# assert exercise_3_all_starts([1, 2, 3, 4, 5], [3, 4, 5, 1, 2]) == [3]
# assert exercise_3_all_starts([2, 3, 4], [3, 4, 3]) == []
# assert exercise_3_all_starts([1, 1, 1], [1, 1, 1]) == [0, 1, 2]

## Bài 4 — Biến thể: chạy ngược chiều kim đồng hồ

Đi từ trạm `i` tới trạm `i − 1` tốn `cost_back[i]`. Tìm điểm xuất phát (bất kỳ nếu nhiều), hoặc `-1`.

In [16]:
def exercise_4_counter_clockwise(gas: List[int], cost_back: List[int]) -> int:
    # TODO
    pass


# s = exercise_4_counter_clockwise([1, 2, 3, 4, 5], [2, 3, 4, 5, 1]); assert s != -1
# assert exercise_4_counter_clockwise([1, 1], [2, 2]) == -1

## Bài 5 — Mini challenge: Maximum Sum Circular Subarray (LeetCode #918)

Tổng lớn nhất của một mảng con **liên tiếp** trên mảng **vòng** (không lấy rỗng).

```python
max_circular([1, -2, 3, -2])  -> 3
max_circular([5, -3, 5])      -> 10
max_circular([-3, -2, -3])    -> -2
```

Gợi ý nhanh: đáp án là `max(Kadane max, total − Kadane min)`; cẩn thận khi mọi phần tử âm.

In [17]:
def exercise_5_max_circular(nums: List[int]) -> int:
    # TODO
    pass


# assert exercise_5_max_circular([1, -2, 3, -2]) == 3
# assert exercise_5_max_circular([5, -3, 5]) == 10
# assert exercise_5_max_circular([-3, -2, -3]) == -2

# 12. Gợi ý

**Bài 1**
- Mức 1: dùng `trace` sau khi tự làm.

**Bài 3**
- Mức 1: điều kiện 1: `Q[s] ≤ min(Q[s+1..n])` — dùng mảng **min hậu tố**.
- Mức 2: điều kiện 2: `min(Q[1..s]) ≥ Q[s] − total` — dùng **min tiền tố**. Cả hai `O(1)` mỗi `s` sau khi dựng mảng.

**Bài 4**
- Mức 1: đi ngược = đảo thứ tự trạm. Với thứ tự mới `j = n−1−i`, xăng `gas[i]`, chi phí `cost_back[i]`. Giải bằng bài chính rồi đổi chỉ số về.

**Bài 5**
- Mức 1: Kadane tìm `max_sub` và `min_sub`; nếu `max_sub < 0` (mọi phần tử âm) trả `max_sub`.

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [18]:
trace([3, 1, 1], [1, 2, 2])
print()
trace([5, 8, 2, 8], [6, 5, 6, 6])

 i  gas cost  diff  tank start
 0    3    1    +2     2     0
 1    1    2    -1     1     0
 2    1    2    -1     0     0
total = 0 -> kết quả 0

 i  gas cost  diff  tank start
 0    5    6    -1     0     1  tank âm -> bỏ 0..0, start = 1
 1    8    5    +3     3     1
 2    2    6    -4     0     3  tank âm -> bỏ 1..2, start = 3
 3    8    6    +2     2     3
total = 0 -> kết quả 3


## Đáp án Bài 2

In [19]:
def answer_2_can_complete(gas: List[int], cost: List[int]) -> int:
    return can_complete(gas, cost)

assert answer_2_can_complete([1, 2, 3, 4, 5], [3, 4, 5, 1, 2]) == 3
assert answer_2_can_complete([2, 3, 4], [3, 4, 3]) == -1
assert answer_2_can_complete([5], [4]) == 0
print("Bài 2 PASS (giống can_complete ở mục 4)")

Bài 2 PASS (giống can_complete ở mục 4)


## Đáp án Bài 3

In [20]:
def answer_3_all_starts(gas: List[int], cost: List[int]) -> List[int]:
    n = len(gas)
    Q = [0] * (n + 1)
    for i in range(n):
        Q[i + 1] = Q[i] + gas[i] - cost[i]
    total = Q[n]
    if total < 0:
        return []
    suf = [0] * (n + 2)                   # suf[t] = min(Q[t..n])
    suf[n + 1] = float("inf")
    for t in range(n, -1, -1):
        suf[t] = min(Q[t], suf[t + 1])
    out, pre = [], float("inf")           # pre = min(Q[1..s])
    for s_ in range(n):
        if s_ >= 1:
            pre = min(pre, Q[s_])
        if Q[s_] <= suf[s_ + 1] and pre >= Q[s_] - total:
            out.append(s_)
    return out

assert answer_3_all_starts([1, 2, 3, 4, 5], [3, 4, 5, 1, 2]) == [3]
assert answer_3_all_starts([2, 3, 4], [3, 4, 3]) == []
assert answer_3_all_starts([1, 1, 1], [1, 1, 1]) == [0, 1, 2]
for _ in range(3000):
    n = random.randint(1, 8)
    gas = [random.randint(0, 5) for _ in range(n)]
    cost = [random.randint(0, 5) for _ in range(n)]
    assert answer_3_all_starts(gas, cost) == [s for s in range(n) if feasible(gas, cost, s)]
print("Bài 3 PASS")

Bài 3 PASS


Điều kiện 1 bảo đảm không cạn xăng trước khi quay vòng; điều kiện 2 bảo đảm không cạn xăng trên đoạn quay vòng `0..s−1`.

## Đáp án Bài 4

In [21]:
def answer_4_counter_clockwise(gas: List[int], cost_back: List[int]) -> int:
    n = len(gas)
    order = list(range(n - 1, -1, -1))         # vị trí mới j ứng với trạm order[j]
    r = can_complete([gas[i] for i in order], [cost_back[i] for i in order])
    return -1 if r == -1 else order[r]

def feasible_back(gas, cost_back, s):
    tank, n = 0, len(gas)
    for k in range(n):
        i = (s - k) % n
        tank += gas[i] - cost_back[i]
        if tank < 0:
            return False
    return True

s = answer_4_counter_clockwise([1, 2, 3, 4, 5], [2, 3, 4, 5, 1]); assert s != -1
assert answer_4_counter_clockwise([1, 1], [2, 2]) == -1
for _ in range(3000):
    n = random.randint(1, 8)
    gas = [random.randint(0, 5) for _ in range(n)]
    cb = [random.randint(0, 5) for _ in range(n)]
    r = answer_4_counter_clockwise(gas, cb)
    exists = any(feasible_back(gas, cb, t) for t in range(n))
    assert (r == -1) == (not exists) and (r == -1 or feasible_back(gas, cb, r))
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [22]:
def answer_5_max_circular(nums: List[int]) -> int:
    best_max = cur_max = nums[0]
    best_min = cur_min = nums[0]
    for x in nums[1:]:
        cur_max = max(x, cur_max + x); best_max = max(best_max, cur_max)
        cur_min = min(x, cur_min + x); best_min = min(best_min, cur_min)
    if best_max < 0:
        return best_max
    return max(best_max, sum(nums) - best_min)

def brute_circular(nums):
    n = len(nums)
    return max(sum(nums[(s + k) % n] for k in range(L)) for s in range(n) for L in range(1, n + 1))

assert answer_5_max_circular([1, -2, 3, -2]) == 3
assert answer_5_max_circular([5, -3, 5]) == 10
assert answer_5_max_circular([-3, -2, -3]) == -2
for _ in range(2000):
    nums = [random.randint(-5, 5) for _ in range(random.randint(1, 8))]
    assert answer_5_max_circular(nums) == brute_circular(nums)
print("Bài 5 PASS")

Bài 5 PASS

# 14. Tổng kết

> `total = Σ(gas − cost)`; `total < 0` ⇒ `-1`. Ngược lại một lượt: cộng dồn `tank`; mỗi khi `tank < 0` thì đặt `start = i + 1`, `tank = 0`. `start` cuối cùng là đáp án. `O(n)`, `O(1)`.

---

## Checklist trước khi nộp

- [ ] Kiểm tra `total >= 0`?
- [ ] Đặt lại `start = i + 1` (không phải `i`)?
- [ ] Không đi vòng lặp lại (một lượt là đủ)?
- [ ] Giải thích được vì sao bỏ qua cả đoạn `s..j` là an toàn?